In [1]:
!pip install "numpy<2.0" "scipy<1.15"

# Now install the rest (removed --force-reinstall to save you 5+ mins of waiting, unless your env is totally broken)
!pip install --upgrade torch torchvision torchaudio transformers accelerate datasets scikit-learn seqeval evaluate

print("✅ Libraries Installed & Synced for T4 GPU.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 49.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.2.2 which is incompatible.
ml

In [2]:
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForSequenceClassification
from datasets import Dataset


from scipy.stats import pearsonr
from tqdm import tqdm
import math
import re
import requests

# XLM-RoBERTa is excellent for this cross-lingual matching task
MODEL_CHECKPOINT = "xlm-roberta-base" 
MAX_LEN = 128

ImportError: cannot import name 'DataCollatorForSequenceClassification' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)

In [ ]:
# def load_jsonl_url(url):
#     try:
#         response = requests.get(url)
#         response.raise_for_status()
#         return [json.loads(line) for line in response.text.strip().split('\n') if line]
#     except Exception as e:
#         print(f"Error loading {url}: {e}")
#         return []

# # --- Load All Data (Eng + Zho) ---
# # Update with your actual paths!
# base_url = "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_1/"
# augmented_url = "https://raw.githubusercontent.com/affan002/DeepLearning_project/refs/heads/main/augmented_datasets/augmented_translation_zho_to_eng"
# files = [
#     f"{base_url}eng/eng_laptop_train_alltasks.jsonl",
#     f"{base_url}eng/eng_restaurant_train_alltasks.jsonl",
#     f"{base_url}zho/zho_laptop_train_alltasks.jsonl",
#     f"{base_url}zho/zho_restaurant_train_alltasks.jsonl",
#     f"{augmented_url}/eng_from_zho_laptop.jsonl",
#     f"{augmented_url}/eng_from_zho_restaurant.jsonl",
#     f"{augmented_url}/zho_from_eng_laptop.jsonl",
#     f"{augmented_url}/zho_from_eng_restaurant.jsonl",

# ]

# all_data = []
# for url in files:
#     all_data.extend(load_jsonl_url(url))

# print(f"Loaded {len(all_data)} raw sentences.")

In [ ]:
import json
import requests
import pandas as pd
import random

# --- 1. HELPER FUNCTIONS ---

def load_jsonl_url(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return [json.loads(line) for line in response.text.strip().split('\n') if line]
    except Exception as e:
        print(f"❌ Error loading JSONL {url}: {e}")
        return []

def load_json_url(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"❌ Error loading JSON {url}: {e}")
        return []

def transform_sighan_data(sighan_list):
    """Converts SIGHAN parallel lists to DimABSA format."""
    transformed = []
    for entry in sighan_list:
        quads = []
        count = len(entry.get("Aspect", []))
        for i in range(count):
            quads.append({
                "Aspect": entry["Aspect"][i],
                "Category": entry["Category"][i],
                "Opinion": entry["Opinion"][i],
                "VA": entry["Intensity"][i]
            })
        
        transformed.append({
            "ID": entry["ID"],
            "Text": entry["Sentence"], 
            "Quadruplet": quads
        })
    return transformed

def filter_data(data_list):
    """
    Strictly filters augmented data. 
    Drops rows where Aspect/Opinion are missing from Text or are None.
    """
    valid_data = []
    dropped = 0
    
    for entry in data_list:
        # Ensure text is a string
        text = str(entry.get('Text', "") or "").lower()
        quads = entry.get('Quadruplet', [])
        
        if not quads: continue
            
        is_valid = True
        for q in quads:
            # FIX: Convert None to empty string safely before lower()
            asp = str(q.get('Aspect') or "").lower()
            opi = str(q.get('Opinion') or "").lower()
            
            # Rule: Words MUST exist in the text
            # Also check if they are empty strings (which means they were None)
            if not asp or (asp != "null" and asp not in text):
                is_valid = False; break
            
            if not opi or (opi != "null" and opi not in text):
                is_valid = False; break
        
        if is_valid:
            valid_data.append(entry)
        else:
            dropped += 1
            
    print(f"   [Filter] Dropped {dropped} misaligned rows.")
    return valid_data

# --- 2. DEFINE URL SOURCES ---

# A. Original
base_url = "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_1/"
original_urls = [
    f"{base_url}eng/eng_laptop_train_alltasks.jsonl",
    f"{base_url}eng/eng_restaurant_train_alltasks.jsonl",
    f"{base_url}zho/zho_laptop_train_alltasks.jsonl",
    f"{base_url}zho/zho_restaurant_train_alltasks.jsonl",
]

B. Synthetic (Llama 8B)
syn_base = "https://raw.githubusercontent.com/affan002/DeepLearning_project/refs/heads/main/augmented_datasets//first_aug_synthetic_llama_8b"
synthetic_urls = [
    f"{syn_base}/augmented_eng_laptop_train_alltasks.jsonl",
    f"{syn_base}/augmented_eng_restaurant_train_alltasks.jsonl",
    f"{syn_base}/augmented_zho_laptop_train_alltasks.jsonl",
    f"{syn_base}/augmented_zho_restaurant_train_alltasks.jsonl",
]



# D. SIGHAN (Chinese)
sighan_urls = [
    "https://raw.githubusercontent.com/NYCU-NLP/SIGHAN2024-dimABSA/refs/heads/main/DataSets/dimABSA2024/Simplified/SIGHAN2024_dimABSA_TrainingSet1_Simplified.json",
    "https://raw.githubusercontent.com/NYCU-NLP/SIGHAN2024-dimABSA/refs/heads/main/DataSets/dimABSA2024/Simplified/SIGHAN2024_dimABSA_TrainingSet2_Simplified.json"
]

# --- 3. EXECUTE LOAD ---
all_data = []

print("--- 1. Loading Original Data ---")
for url in original_urls:
    data = load_jsonl_url(url)
    all_data.extend(data)
    print(f"   Loaded {len(data)} samples.")

print("\n--- 2. Loading & Filtering SIGHAN ---")
for url in sighan_urls:
    data = load_json_url(url)
    if data:
        clean = transform_sighan_data(data)
        all_data.extend(clean)
        print(f"   Added {len(clean)} SIGHAN samples.")

print("\n--- 3. Loading & Filtering Synthetic (Llama 8B) ---")
for url in synthetic_urls:
    data = load_jsonl_url(url)
    if data:
        clean = filter_data(data)
        all_data.extend(clean)
        print(f"   Added {len(clean)} valid synthetic samples.")


print(f"\n✅ TOTAL TRAINING POOL: {len(all_data)} sentences.")

In [ ]:
# def create_pairing_dataset(data_list):
#     """
#     Transforms sentences into (Sentence, Aspect, Opinion, Label) pairs.
#     Generates HARD NEGATIVES by mismatching aspects/opinions within the same sentence.
#     """
#     dataset_rows = []
    
#     for entry in data_list:
#         text = entry.get('Text', '')
#         quads = entry.get('Quadruplet', entry.get('Triplet', entry.get('Aspect_VA', [])))
#         if not isinstance(quads, list): continue
            
#         # 1. Collect Valid Pairs (Ground Truth)
#         valid_pairs = set()
#         all_aspects = set()
#         all_opinions = set()
        
#         for q in quads:
#             a = q.get('Aspect')
#             o = q.get('Opinion')
            
#             # We only care if both exist. (Task 2 requires both)
#             if a and o and a != 'NULL' and o != 'NULL':
#                 valid_pairs.add((a, o))
#                 all_aspects.add(a)
#                 all_opinions.add(o)
                
#                 # --- POSITIVE SAMPLE (Label = 1) ---
#                 dataset_rows.append({
#                     "text": text,
#                     "aspect": a,
#                     "opinion": o,
#                     "label": 1
#                 })
        
#         # 2. Generate Negative Pairs (Hard Negatives)
#         # We take the Cartesian Product of all Aspects x all Opinions in this sentence
#         # If a combination is NOT in valid_pairs, it is a Negative.
        
#         for a in all_aspects:
#             for o in all_opinions:
#                 if (a, o) not in valid_pairs:
#                     # --- NEGATIVE SAMPLE (Label = 0) ---
#                     # Example: "Food was good but service was slow."
#                     # Pair: (Food, slow) -> INVALID
#                     dataset_rows.append({
#                         "text": text,
#                         "aspect": a,
#                         "opinion": o,
#                         "label": 0
#                     })

#     return pd.DataFrame(dataset_rows)

# # --- Execute ---
# print("Generating Positive and Negative Pairs...")
# pair_df = create_pairing_dataset(all_data)

# print(f"Total Pairs Generated: {len(pair_df)}")
# print("Class Distribution:")
# print(pair_df['label'].value_counts())

# # Stratified Split (Balanced)
# train_df, temp_dev_df = train_test_split(
#     pair_df, 
#     test_size=0.2, 
#     random_state=42, 
#     stratify=pair_df['label']
# )
# dev_df, test_df = train_test_split(temp_dev_df, test_size=0.5, random_state=42)
# print(f"Train Sentences: {len(train_df)} | Val Sentences: {len(dev_df)} | Test Sentences: {len(test_df)}")

In [ ]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split

# --- 1. Robust Data Loader ---
def create_pairing_dataset(data_list, target_ratio=1.0):
    """
    Generates Positive pairs (Ground Truth) and Negative pairs.
    Strategy: 
      1. Hard Negatives: Aspect + Wrong Opinion (Same Sentence)
      2. Soft Negatives: Aspect + Random Opinion (Different Sentence)
    target_ratio: 1.0 means we generate 1 Negative for every 1 Positive.
    """
    features = []
    
    # First pass: Collect all valid opinions globally for random sampling
    all_global_opinions = []
    for entry in data_list:
        quads = entry.get('Quadruplet', entry.get('Triplet', []))
        if isinstance(quads, list):
            for q in quads:
                if q.get('Opinion') and q.get('Opinion') != 'NULL':
                    all_global_opinions.append(q.get('Opinion'))
    
    # Second pass: Create pairs
    for entry in data_list:
        text = entry.get('Text', '')
        quads = entry.get('Quadruplet', entry.get('Triplet', []))
        if not isinstance(quads, list): continue
            
        valid_pairs = set()
        local_aspects = set()
        local_opinions = set()
        
        # --- A. POSITIVES (Valid Pairs) ---
        for q in quads:
            a = q.get('Aspect')
            o = q.get('Opinion')
            if a and o and a != 'NULL' and o != 'NULL':
                valid_pairs.add((a, o))
                local_aspects.add(a)
                local_opinions.add(o)
                
                features.append({
                    "text": text, "aspect": a, "opinion": o, "label": 1 
                })

        # --- B. NEGATIVES (Invalid Pairs) ---
        num_positives = len(valid_pairs)
        num_negatives_needed = int(num_positives * target_ratio)
        current_negatives = 0
        
        # B1. Hard Negatives (Same Sentence Mismatches)
        for a in local_aspects:
            for o in local_opinions:
                if (a, o) not in valid_pairs and current_negatives < num_negatives_needed:
                    features.append({
                        "text": text, "aspect": a, "opinion": o, "label": 0 
                    })
                    current_negatives += 1
        
        # B2. Soft Negatives (Randomly Sampled from Global List)
        # Fills the gap if the sentence didn't have enough internal mismatches
        attempts = 0
        while current_negatives < num_negatives_needed and attempts < 20:
            random_o = random.choice(all_global_opinions)
            if not local_aspects: break
            random_a = random.choice(list(local_aspects))
            
            if (random_a, random_o) not in valid_pairs:
                features.append({
                    "text": text, "aspect": random_a, "opinion": random_o, "label": 0 
                })
                current_negatives += 1
            attempts += 1
                    
    return pd.DataFrame(features)

# --- EXECUTE ---
print("Generating Balanced Pairing Data...")
# We use target_ratio=1.0 to force a 50/50 split
pair_df = create_pairing_dataset(all_data, target_ratio=1.0)

print(f"Total Pairs: {len(pair_df)}")
print("Class Distribution (Should be roughly equal):")
print(pair_df['label'].value_counts())

print("\nCreating 3-Way Split (Train / Val / Test)...")

# Split 1: Train (80%) vs Temp (20%)
train_df, temp_dev_df = train_test_split(
    pair_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=pair_df['label'] # Ensure balanced 0s and 1s
)

# Split 2: Temp (20%) -> Dev (10%) + Test (10%)
# We split the remaining 20% in half
dev_df, test_df = train_test_split(
    temp_dev_df, 
    test_size=0.5, 
    random_state=42,
    stratify=temp_dev_df['label'] # Ensure balanced 0s and 1s
)

print(f"Train Pairs: {len(train_df)}")
print(f"Val Pairs:   {len(dev_df)}")
print(f"Test Pairs:  {len(test_df)}")

In [ ]:
train_df.head(5)


In [ ]:
NULL_TOKEN = "[NULL]"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
tokenizer.add_special_tokens({'additional_special_tokens': [NULL_TOKEN]})

def preprocess_pairing(examples):
    # 1. Prepare Input A (Sentence with Double NULL)
    # This aligns with what the Extractor saw
    sentences = [f"{NULL_TOKEN} {NULL_TOKEN} {t}" for t in examples["text"]]
    
    # 2. Prepare Input B (Aspect + Opinion Pair)
    # We join them with the separator token
    pairs = [f"{a} {tokenizer.sep_token} {o}" for a, o in zip(examples["aspect"], examples["opinion"])]
    
    return tokenizer(
        sentences,       # Input A
        pairs,           # Input B
        truncation=True,
        padding="max_length", 
        max_length=MAX_LEN
    )

# Create HF Datasets
hf_train = Dataset.from_pandas(train_df).map(preprocess_pairing, batched=True)
hf_dev = Dataset.from_pandas(dev_df).map(preprocess_pairing, batched=True)
hf_test = Dataset.from_pandas(test_df).map(preprocess_pairing, batched=True)

hf_train = hf_train.rename_column("label", "labels")
hf_dev = hf_dev.rename_column("label", "labels")
hf_test = hf_test.rename_column("label", "labels")

# Cleanup columns for PyTorch
cols = ['input_ids', 'attention_mask', 'labels']
hf_train = hf_train.select_columns(cols)
hf_dev = hf_dev.select_columns(cols)
hf_test = hf_test.select_columns(cols)

In [ ]:
import random

def check_tokenization(dataset, tokenizer, num_samples=3):
    # Pick random samples
    indices = random.sample(range(len(dataset)), num_samples)
    
    print(f"--- Checking {num_samples} Random Samples ---")
    
    for idx in indices:
        example = dataset[idx]
        input_ids = example['input_ids']
        label = example['labels']
        
        # Decode the IDs back to text
        decoded_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        
        print(f"\nSample {idx}:")
        print(f"Label: {label} ({'Valid' if label==1 else 'Invalid'})")
        print(f"Input IDs Length: {len(input_ids)}")
        print(f"Decoded: {decoded_text}")
        print("-" * 40)

# Run the check
check_tokenization(hf_train, tokenizer)

In [ ]:
# Check Label Balance
labels = hf_train['labels']
ones = sum(labels)
zeros = len(labels) - ones
print(f"Training Data Balance: {zeros} Negatives (0) vs {ones} Positives (1)")

# Check a random input sample
import random
idx = random.randint(0, len(hf_train)-1)
print(f"\nSample Input (Decoded):")
print(tokenizer.decode(hf_train[idx]['input_ids'], skip_special_tokens=False))
print(f"Label: {hf_train[idx]['labels']}")

In [ ]:
# import evaluate
import sklearn.metrics as calc_metrics 
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, DataCollatorForTokenClassification, DataCollatorWithPadding
from tqdm.auto import tqdm
import numpy as np

BATCH_SIZE=16
LR = 2e-5
EPOCHS = 10

In [ ]:
from transformers import AutoModelForSequenceClassification
# --- 1. Setup Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2 
)
model.resize_token_embeddings(len(tokenizer)) # CRITICAL RESIZE
model.to(device)

# --- 2. Loaders & Optimizer ---
data_collator = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(hf_train, shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator)
val_loader = DataLoader(hf_dev, batch_size=BATCH_SIZE, collate_fn=data_collator)



loss_fct = torch.nn.CrossEntropyLoss()

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * EPOCHS * len(train_loader)), 
    num_training_steps=EPOCHS * len(train_loader)
)

# --- 3. Metrics Helper ---
def calculate_metrics(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return {
        "acc": calc_metrics.accuracy_score(labels_flat, preds_flat),
        "f1": calc_metrics.f1_score(labels_flat, preds_flat, average="binary"),
        "precision": calc_metrics.precision_score(labels_flat, preds_flat, average="binary"),
        "recall": calc_metrics.recall_score(labels_flat, preds_flat, average="binary")
    }

# --- 4. Execution ---
print(f"Starting Training on {device}...")
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_val_acc = 0

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = loss_fct(outputs.logits, batch["labels"])
        loss.backward()

        if total_train_loss == 0: 
            probs = torch.softmax(outputs.logits, dim=1)
            print(f"\nDEBUG: First Batch Preds: {probs[0:5].detach().cpu().numpy()}")
            
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    history["train_loss"].append(avg_train_loss)

    # EVALUATE
    model.eval()
    total_val_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Val"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = loss_fct(outputs.logits, batch["labels"])
            total_val_loss += loss.item()
            all_preds.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    history["val_loss"].append(avg_val_loss)
    
    # Calc Metrics
    all_preds = np.vstack(all_preds)
    all_labels = np.concatenate(all_labels)
    cur_metrics = calculate_metrics(all_preds, all_labels)
    
    history["val_acc"].append(cur_metrics["acc"])
    history["val_f1"].append(cur_metrics["f1"])
    
    
    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f} | Acc={cur_metrics['acc']:.4f}, F1={cur_metrics['f1']:.4f}, Precision={cur_metrics['precision']:.4f}, Recall={cur_metrics['recall']:.4f}")

    # Save Best
    if cur_metrics["acc"] > best_val_acc:
        best_val_acc = cur_metrics["acc"]
        model.save_pretrained("final_pairing_model")
        tokenizer.save_pretrained("final_pairing_model")
        print("✅ New Best Model Saved!")


In [ ]:
import matplotlib.pyplot as plt 

# --- 5. Visualization (Elbow Plot) ---
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", marker='o')
plt.plot(history["val_loss"], label="Val Loss", marker='o')
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(history["val_acc"], label="Val Accuracy", color="green", marker='o')
plt.plot(history["val_f1"], label="Val F1", color="purple", marker='x')
plt.title("Performance Metrics")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# --- 1. Evaluate on Test Set ---
# Assuming 'dev_df' (or test_df) was created earlier
print("Evaluating on Validation/Test Set...")
test_loader = DataLoader(
    hf_test, 
    batch_size=BATCH_SIZE, 
    collate_fn=data_collator
)
best_model_path = "final_pairing_model"
loaded_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
loaded_model.to(device)
loaded_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_val_loss += outputs.loss.item()
        all_preds.append(outputs.logits.cpu().numpy())
        all_labels.append(batch["labels"].cpu().numpy())

# --- 4. Calculate & Print Final Metrics ---
all_preds = np.vstack(all_preds)
all_labels = np.concatenate(all_labels)
test_metrics = calculate_metrics(all_preds, all_labels)

print("\n" + "="*40)
print("🏁 FINAL TEST SET RESULTS")
print("="*40)
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1 Score:  {test_metrics['f1']:.4f}")
print(f"Accuracy:  {test_metrics['acc']:.4f}")
print("="*40)


## Manual testing 

In [ ]:
# Ensure NULL_TOKEN is defined
NULL_TOKEN = "[NULL]" 

def check_pair(text, aspect, opinion):
    # --- FIX: Add the [NULL] tokens to match training data ---
    augmented_text = f"{NULL_TOKEN} {NULL_TOKEN} {text}"
    
    # Format input B: "Aspect </s> Opinion"
    pair_text = f"{aspect} {tokenizer.sep_token} {opinion}"
    
    tokenized = tokenizer(
        augmented_text,      # Input A (Sentence with NULLs)
        pair_text,           # Input B (Pair)
        truncation=True, 
        padding="max_length", 
        max_length=128,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = loaded_model(**tokenized)
        
    probs = torch.softmax(outputs.logits, dim=1)[0]
    prediction = torch.argmax(probs).item()
    score = probs[1].item() # Probability of being Valid
    
    status = "VALID" if prediction == 1 else "INVALID"
    # Print with high precision to see confidence
    print(f"Pair: ({aspect}, {opinion}) -> {status}  (Confidence: {score:.4f})")

print("\n--- Manual Tests ---")
sent = "The food was delicious but the service was terrible."
print(f"Sentence: {sent}\n")

# These should have High Confidence (> 0.9)
check_pair(sent, "food", "delicious")   
check_pair(sent, "service", "terrible") 

# These should have Low Confidence (< 0.1)
check_pair(sent, "food", "terrible")    
check_pair(sent, "service", "delicious")

## Saving the model on HF

In [ ]:
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

hf_token = os.environ["HF_TOKEN"]
login(token=hf_token, write_permission=True)


# --- 2. Push to Hub ---
# Change this to your desired name
REPO_NAME = "affan002/xlm-roberta-pairing-eng-zho-aug-synthetic" 

if hf_token:
    print(f"🚀 Pushing model to Hugging Face: {REPO_NAME}...")
    
    # Push Model weights
    model.push_to_hub(
        REPO_NAME, 
        commit_message="Training complete - Pairing Model (Negative Sampling)",
        token=hf_token
    )
    
    # Push Tokenizer (Critical for [NULL] token support)
    tokenizer.push_to_hub(
        REPO_NAME, 
        commit_message="Tokenizer with [NULL] token",
        token=hf_token
    )
    
    print("✅ Success! Model is live on the Hub.")